# MeetScribe - Trascrizione riunioni con GPU

Questo notebook esegue la pipeline MeetScribe completa su Google Colab con GPU T4.

**Prima di iniziare:**
1. Vai su `Runtime > Change runtime type > T4 GPU`
2. Avrai bisogno del tuo file audio e del token HuggingFace

## 1. Setup: installa MeetScribe + dipendenze GPU

In [ ]:
import os

# Clona il repo (sostituisci con il tuo URL)
REPO_URL = 'https://github.com/paratusapp/meet-scribe.git'  # <-- CAMBIA QUI

repo_dir = '/content/meet-scribe'
if not os.path.exists(repo_dir):
    !git clone {REPO_URL} {repo_dir}
else:
    print('Repo già clonato')
%cd {repo_dir}

# Installa il progetto
!pip install -e . -q

# Verifica GPU
import torch
print(f"\nGPU disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configura il token HuggingFace

Serve per scaricare i modelli pyannote (speaker diarization).
Prendi il token da: https://huggingface.co/settings/tokens

In [ ]:
import os
from google.colab import userdata

# Opzione 1: usa Colab Secrets (consigliato)
# Vai su icona chiave nella sidebar > aggiungi 'HF_TOKEN' con il tuo token
try:
    os.environ['HUGGING_FACE_TOKEN'] = userdata.get('HF_TOKEN')
    print('Token caricato da Colab Secrets')
except Exception:
    # Opzione 2: incolla direttamente (meno sicuro)
    os.environ['HUGGING_FACE_TOKEN'] = 'hf_YOUR_TOKEN_HERE'  # <-- sostituisci
    print('Token impostato manualmente')

## 3. Carica il file audio

Carica il tuo file audio (.m4a, .mp4, .wav, ecc.)

In [ ]:
from google.colab import files
from pathlib import Path

# Upload del file audio
print('Seleziona il file audio da trascrivere...')
uploaded = files.upload()

# Salva nella cartella recordings
recordings_dir = Path('recordings')
recordings_dir.mkdir(exist_ok=True)

for filename, content in uploaded.items():
    audio_path = recordings_dir / filename
    audio_path.write_bytes(content)
    print(f'\nFile salvato: {audio_path}')
    print(f'Dimensione: {len(content) / 1e6:.1f} MB')

## 4. Esegui MeetScribe

In [ ]:
# Esegui la pipeline completa
# Modifica il nome del file e la lingua se necessario
!python -m meet_scribe.main --input recordings/rec-1.m4a --lang en

## 5. Visualizza e scarica i risultati

In [ ]:
from pathlib import Path

# Mostra il file TXT
output_dir = Path('output')
for txt_file in output_dir.glob('*.txt'):
    print(f'=== {txt_file.name} ===')
    print(txt_file.read_text(encoding='utf-8')[:3000])
    print('...(troncato per display)\n')

In [ ]:
# Scarica i file di output
from google.colab import files
from pathlib import Path

output_dir = Path('output')
for f in output_dir.iterdir():
    print(f'Download: {f.name}')
    files.download(str(f))